In [1]:
import json
import os
from collections import defaultdict

In [2]:
INPUT_DIR = "/kaggle/input/notebooks/muhammadnabilul/04-chunking-and-splitting/chunks"

chunks = []
for fname in sorted(f for f in os.listdir(INPUT_DIR) if f.endswith(".json")):
    data = json.load(open(os.path.join(INPUT_DIR, fname), encoding="utf-8"))
    chunks.extend(data if isinstance(data, list) else [data])
 
print(f"Total chunks: {len(chunks)}")

Total chunks: 23336


In [3]:
# Rekonstruksi section 
sections = {}
for chunk in chunks:
    key = (chunk.get("document_title"), chunk.get("section_id"))
    if key not in sections:
        sections[key] = {
            "document_title":    chunk.get("document_title"),
            "section_id":        chunk.get("section_id"),
            "section_source":    chunk.get("section_source", ""),
            "section_chunks":    [],
            "section_token_est": 0,
        }
    sections[key]["section_chunks"].append({
        "chunk_id":       chunk["chunk_id"],
        "text":           chunk.get("text", ""),
        "chunk_type":     chunk.get("chunk_type", "text_only"),
        "token_estimate": chunk.get("token_estimate", 0),
        "page_start":     chunk.get("chunk_page_start", 0),
        "page_end":       chunk.get("chunk_page_end", 0),
    })
    sections[key]["section_token_est"] += chunk.get("token_estimate", 0)
 
print(f"Total sections: {len(sections)}")

Total sections: 7799


In [4]:
# Simulasi batch_section untuk setiap section 
MAX_TOKENS   = 1500
OVERLAP      = 150
STEP         = MAX_TOKENS - OVERLAP
WORDS_PER_TOK = 0.74
MAX_WORDS    = int(MAX_TOKENS * WORDS_PER_TOK)
STEP_WORDS   = int(STEP * WORDS_PER_TOK)
 
batch_records = []
 
for key, sec in sections.items():
    word_stream = []
    for sc in sec["section_chunks"]:
        pages = list(range(sc["page_start"], sc["page_end"] + 1))
        for word in sc["text"].split():
            word_stream.append({"chunk_id": sc["chunk_id"], "pages": pages})
 
    if not word_stream:
        continue
 
    pos = 0
    while pos < len(word_stream):
        window      = word_stream[pos: pos + MAX_WORDS]
        chunk_ids   = list(dict.fromkeys(w["chunk_id"] for w in window))
        token_count = max(1, int(len(window) / WORDS_PER_TOK * WORDS_PER_TOK))
        token_count = max(1, int(len(window) * (1 / WORDS_PER_TOK) * WORDS_PER_TOK))
        token_count = max(1, int(len(window) / WORDS_PER_TOK))
 
        batch_records.append({
            "section_id":          key[1],
            "document_title":      sec["document_title"],
            "section_source":      sec["section_source"],
            "context_token_count": token_count,
            "chunk_ids_count":     len(chunk_ids),
        })
        pos += STEP_WORDS
 
print(f"Total batches simulasi: {len(batch_records)}")

Total batches simulasi: 9350


In [5]:
# Statistik token per batch 
tokens = [b["context_token_count"] for b in batch_records]
tokens_sorted = sorted(tokens)
n = len(tokens_sorted)
 
def percentile(data, p):
    idx = int(len(data) * p / 100)
    return data[min(idx, len(data)-1)]
 
print("\n── Distribusi context_token_count ──────────────────")
print(f"  Min    : {min(tokens)}")
print(f"  P10    : {percentile(tokens_sorted, 10)}")
print(f"  P25    : {percentile(tokens_sorted, 25)}")
print(f"  Median : {percentile(tokens_sorted, 50)}")
print(f"  P75    : {percentile(tokens_sorted, 75)}")
print(f"  P90    : {percentile(tokens_sorted, 90)}")
print(f"  P95    : {percentile(tokens_sorted, 95)}")
print(f"  Max    : {max(tokens)}")
print(f"  Mean   : {sum(tokens)/n:.1f}")


── Distribusi context_token_count ──────────────────
  Min    : 1
  P10    : 94
  P25    : 220
  Median : 401
  P75    : 835
  P90    : 1500
  P95    : 1500
  Max    : 1500
  Mean   : 587.2


In [6]:
# Distribusi chunk_ids_count per batch 
cids = [b["chunk_ids_count"] for b in batch_records]
from collections import Counter
cid_dist = Counter(cids)
print("\n── Distribusi chunk_ids_count per batch ────────────")
for k in sorted(cid_dist):
    pct = cid_dist[k] / n * 100
    print(f"  {k} chunk(s): {cid_dist[k]:5d} ({pct:5.1f}%) ")


── Distribusi chunk_ids_count per batch ────────────
  1 chunk(s):  4267 ( 45.6%) 
  2 chunk(s):  1516 ( 16.2%) 
  3 chunk(s):  1013 ( 10.8%) 
  4 chunk(s):   680 (  7.3%) 
  5 chunk(s):   576 (  6.2%) 
  6 chunk(s):   411 (  4.4%) 
  7 chunk(s):   380 (  4.1%) 
  8 chunk(s):   234 (  2.5%) 
  9 chunk(s):   155 (  1.7%) 
  10 chunk(s):    75 (  0.8%) 
  11 chunk(s):    28 (  0.3%) 
  12 chunk(s):    10 (  0.1%) 
  13 chunk(s):     2 (  0.0%) 
  14 chunk(s):     1 (  0.0%) 
  15 chunk(s):     1 (  0.0%) 
  17 chunk(s):     1 (  0.0%) 


In [7]:
#  Histogram token (bucket 100) 
print("\n── Histogram context_token_count (bucket 100) ──────")
buckets = defaultdict(int)
for t in tokens:
    buckets[(t // 100) * 100] += 1
for lo in sorted(buckets):
    hi  = lo + 99
    cnt = buckets[lo]
    pct = cnt / n * 100
    print(f"  {lo:4d}–{hi:4d}: {cnt:5d} ({pct:5.1f}%) ")


── Histogram context_token_count (bucket 100) ──────
     0–  99:   959 ( 10.3%) 
   100– 199:  1093 ( 11.7%) 
   200– 299:  1122 ( 12.0%) 
   300– 399:  1476 ( 15.8%) 
   400– 499:   894 (  9.6%) 
   500– 599:   608 (  6.5%) 
   600– 699:   457 (  4.9%) 
   700– 799:   302 (  3.2%) 
   800– 899:   286 (  3.1%) 
   900– 999:   165 (  1.8%) 
  1000–1099:   162 (  1.7%) 
  1100–1199:   130 (  1.4%) 
  1200–1299:   103 (  1.1%) 
  1300–1399:   103 (  1.1%) 
  1400–1499:    91 (  1.0%) 
  1500–1599:  1399 ( 15.0%) 


In [8]:
# Natural breaks kandidat threshold
print("\n── Kandidat threshold (untuk classify_context) ─────")
for p in [25, 50, 75, 90]:
    print(f"  P{p:2d} = {percentile(tokens_sorted, p)} token")


── Kandidat threshold (untuk classify_context) ─────
  P25 = 220 token
  P50 = 401 token
  P75 = 835 token
  P90 = 1500 token


In [9]:
# Simpan raw data untuk analisis lebih lanjut 
out_path = "chunk_distribution.json"
json.dump({
    "total_chunks":  len(chunks),
    "total_sections": len(sections),
    "total_batches": n,
    "stats": {
        "min": min(tokens), "max": max(tokens), "mean": round(sum(tokens)/n, 1),
        "p10": percentile(tokens_sorted, 10),
        "p25": percentile(tokens_sorted, 25),
        "p50": percentile(tokens_sorted, 50),
        "p75": percentile(tokens_sorted, 75),
        "p90": percentile(tokens_sorted, 90),
        "p95": percentile(tokens_sorted, 95),
    },
    "chunk_ids_distribution": {str(k): v for k, v in sorted(cid_dist.items())},
    "token_histogram":        {str(k): v for k, v in sorted(buckets.items())},
}, open(out_path, "w"), indent=2)
print(f"\nRaw data disimpan ke: {out_path}")
print("Paste output di atas ke chat → kita tentukan threshold bersama.")


Raw data disimpan ke: chunk_distribution.json
Paste output di atas ke chat → kita tentukan threshold bersama.
